# Provisionnement - Assurance Non Vie

L'objectif de ce projet est d'appliquer différentes méthodes de provisionnement à un jeu de données acturiels issu du site [casact.org](https://www.casact.org/publications-research/research/research-resources/loss-reserving-data-pulled-naic-schedule-p).

# Sommaire

* [Packages](#partie1)
* [Import des données](#partie2)
* [Mise en forme des données: Triangle de liquidation](#partie3)
* [Méthode de référence: Approche Chain Ladder](#partie4)
    * [Estimation déterministe de la provision](#partie41)
    * [Intervalle de confiance: Méthode de Mack](#partie42)
    * [Distribution de la provision: Boostrap](#partie43)
* [Vérification des hypothèses](#partie5)
* [Backtesting: RMSE - MAE - MAPE](#partie6)

## Packages <a id="partie1"></a>

In [1]:
# %pip install chainladder

import locale
import pandas as pd
import numpy as np
import chainladder as cl
import matplotlib.pyplot as plt

%matplotlib inline

## Import des données <a id="partie2"></a>

In [2]:
df = pd.read_csv("wkcomp_pos_98-07.csv")
df

,GRCODE,GRNAME,AccidentYear,DevelopmentYear,DevelopmentLag,IncurredLosses,CumPaidLoss,BulkLoss,EarnedPremDIR,EarnedPremCeded,EarnedPremNet,Single,PostedReserves2007
0,86,Allstate Ins Co Grp,1998,1998,1,10079,1201,1518,7133,-860,7993,0,135699.214
1,86,Allstate Ins Co Grp,1998,1999,2,9643,2652,240,7133,-860,7993,0,135699.214
2,86,Allstate Ins Co Grp,1998,2000,3,3207,3172,13,7133,-860,7993,0,135699.214
3,86,Allstate Ins Co Grp,1998,2001,4,3202,3178,8,7133,-860,7993,0,135699.214
4,86,Allstate Ins Co Grp,1998,2002,5,3197,3180,8,7133,-860,7993,0,135699.214
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12095,44300,Tower Ins Co Of NY,2005,2010,6,2919,2834,-205,10758,2820,7939,1,18540.838
12096,44300,Tower Ins Co Of NY,2005,2011,7,2809,2899,-318,10758,2820,7939,1,18540.838
12097,44300,Tower Ins Co Of NY,2005,2012,8,3001,2963,-240,10758,2820,7939,1,18540.838
12098,44300,Tower Ins Co Of NY,2005,2013,9,3207,2992,19,10758,2820,7939,1,18540.838


## Mise en forme des données: Triangle de liquidation <a id="partie3"></a>

In [3]:
# Certaines méthodes du package chainladder ne fonctionnent qu'avec une horloge 
# anglophone sur la machine. On regle donc ce probleme manuellement dans le notebook. 

locale.setlocale(locale.LC_TIME, 'English_United States.1252')
print(locale.getlocale(locale.LC_TIME))

('English_United States', '1252')


In [4]:
# Chargement des sinsitres survenus (IncurredLosses) avec Chainladder

bulk_data = cl.Triangle(
    df,
    origin="AccidentYear",
    development="DevelopmentYear",
    columns="IncurredLosses",
    index=["GRNAME"]
)
bulk_data

c:\Users\Onel\AppData\Local\Programs\Python\Python314\Lib\site-packages\chainladder\core\triangle.py:548: UserWarning: 
                The cumulative property of your triangle is not set. This may result in
                undesirable behavior. In a future release this will result in an error.
                
  warnings.warn(


,Triangle Summary
Valuation:,2016-12
Grain:,OYDY
Shape:,"(132, 1, 19, 19)"
Index:,[GRNAME]
Columns:,[IncurredLosses]


In [5]:
# Sommes des montants sur l'index "GRNAME" et passage à des valeurs cumulatives

data = bulk_data.sum().loc[:, :, "1998":"2007", :120].incr_to_cum().dropna()
data

,12,24,36,48,60,72,84,96,108,120
1998,"1,474,384","2,956,897","4,407,489","5,834,174","7,250,711","8,666,008","10,094,283","11,531,774","12,983,881","14,438,098"
1999,"1,544,524","3,081,128","4,632,230","6,197,068","7,773,405","9,379,374","11,028,768","12,683,132","14,385,367","16,088,915"
2000,"1,623,212","3,319,103","5,037,891","6,775,389","8,535,552","10,316,198","12,115,135","13,920,190","15,735,471","17,552,132"
2001,"1,821,092","3,648,373","5,511,198","7,388,004","9,278,363","11,181,447","13,123,219","15,072,576","17,037,394","19,014,458"
2002,"1,970,128","4,015,975","6,060,973","8,108,033","10,153,985","12,186,107","14,218,726","16,250,405","18,281,436","20,306,770"
2003,"2,361,158","4,661,459","6,907,721","9,125,479","11,302,482","13,481,827","15,610,153","17,732,002","19,852,667","21,980,285"
2004,"2,693,419","5,229,679","7,661,394","10,005,005","12,301,674","14,578,511","16,838,091","19,086,405","21,338,986","23,591,838"
2005,"2,843,577","5,469,101","7,967,425","10,408,980","12,803,562","15,171,279","17,531,805","19,891,700","22,264,625","24,610,018"
2006,"2,810,373","5,528,934","8,146,451","10,698,887","13,205,810","15,693,292","18,166,404","20,648,326","23,106,153","25,537,981"
2007,"2,389,599","4,754,680","7,065,131","9,319,718","11,546,546","13,762,231","15,974,789","18,172,772","20,365,959","22,551,770"


In [6]:
# Séparation du triangle supérieur (données d'entrainement)
# Le triangle inférieur servira de données de test pour les méthodes appliquées

triangle_sup = data[data.valuation <= pd.to_datetime('2008-12-31')]
triangle_sup

,12,24,36,48,60,72,84,96,108,120
1998,"1,474,384","2,956,897","4,407,489","5,834,174","7,250,711","8,666,008","10,094,283","11,531,774","12,983,881","14,438,098"
1999,"1,544,524","3,081,128","4,632,230","6,197,068","7,773,405","9,379,374","11,028,768","12,683,132","14,385,367",
2000,"1,623,212","3,319,103","5,037,891","6,775,389","8,535,552","10,316,198","12,115,135","13,920,190",,
2001,"1,821,092","3,648,373","5,511,198","7,388,004","9,278,363","11,181,447","13,123,219",,,
2002,"1,970,128","4,015,975","6,060,973","8,108,033","10,153,985","12,186,107",,,,
2003,"2,361,158","4,661,459","6,907,721","9,125,479","11,302,482",,,,,
2004,"2,693,419","5,229,679","7,661,394","10,005,005",,,,,,
2005,"2,843,577","5,469,101","7,967,425",,,,,,,
2006,"2,810,373","5,528,934",,,,,,,,
2007,"2,389,599",,,,,,,,,


In [7]:
triangle_sup.link_ratio.heatmap()

,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1998,2.0055,1.4906,1.3237,1.2428,1.1952,1.1648,1.1424,1.1259,1.1120
1999,1.9949,1.5034,1.3378,1.2544,1.2066,1.1759,1.1500,1.1342,
2000,2.0448,1.5178,1.3449,1.2598,1.2086,1.1744,1.1490,,
2001,2.0034,1.5106,1.3405,1.2559,1.2051,1.1737,,,
2002,2.0384,1.5092,1.3377,1.2523,1.2001,,,,
2003,1.9742,1.4819,1.3211,1.2386,,,,,
2004,1.9417,1.4650,1.3059,,,,,,
2005,1.9233,1.4568,,,,,,,
2006,1.9673,,,,,,,,
